# arcemx LoRA fine-tune (blueprint 13)

Trains a small QLoRA adapter on arcemx's own graded prediction history,
then exports a merged Q4_K_M GGUF for CPU batch eval in GitHub Actions.

**Monthly loop:**
1. Locally: `python -m analyzer.finetune_export` -> writes `data/finetune/train.jsonl` + `eval.jsonl`
2. Upload both files as a **Kaggle private dataset** (repo is public - never commit these)
3. Attach that dataset to this notebook, set `TRAIN_JSONL_PATH` / `EVAL_JSONL_PATH` below to its mount path
4. Run all cells (needs a Kaggle GPU session - Settings -> Accelerator -> GPU T4 x2 or P100)
5. Download `arcemx_specialist.gguf` from the notebook's Output tab
6. Attach it to a new GitHub Release on the arcemx repo
7. Dispatch `.github/workflows/specialist_eval.yml` with the release tag

No always-on serving (researched: not viable free, July 2026). The GGUF
only ever runs as a scheduled/dispatched CPU batch job.

In [ ]:
# --- PARAMETERS (edit these, nothing else needs changing) ---
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"  # swap to Qwen3-4B or SmolLM3-3B in run 2
TRAIN_JSONL_PATH = "/kaggle/input/arcemx-finetune-data/train.jsonl"
EVAL_JSONL_PATH = "/kaggle/input/arcemx-finetune-data/eval.jsonl"
OUTPUT_DIR = "/kaggle/working/arcemx_specialist"
GGUF_QUANT = "q4_k_m"
MAX_SEQ_LENGTH = 2048

# LoRA hyperparameters (blueprint 13's own decisions, do not tune without
# re-reading the blueprint's reasoning - dataset is small, overfitting risk)
LORA_R = 16
LORA_ALPHA = 16
LEARNING_RATE = 2e-4
LR_SCHEDULER = "cosine"
MIN_LR_RATIO = 5e-5 / 2e-4  # cosine decays 2e-4 -> 5e-5
NUM_EPOCHS = 2  # 1-2 ONLY at this dataset size (few hundred examples)
GENERAL_DATA_FRACTION = 0.08  # 5-10% alpaca-cleaned mixed in, prevents capability collapse

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
import json
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# Load arcemx's own graded-history dataset (temporal split, done by
# finetune_export.py - do NOT reshuffle across the train/eval boundary).
train_ds = load_dataset("json", data_files=TRAIN_JSONL_PATH, split="train")
eval_ds = load_dataset("json", data_files=EVAL_JSONL_PATH, split="train")

# 5-10% general instruction data mixed in to prevent capability collapse
# on this small, narrow-domain dataset.
n_general = int(len(train_ds) * GENERAL_DATA_FRACTION / (1 - GENERAL_DATA_FRACTION))
alpaca = load_dataset("yahma/alpaca-cleaned", split="train").shuffle(seed=42).select(range(n_general))

def alpaca_to_messages(row):
    user = row["instruction"] + (("\n\n" + row["input"]) if row.get("input") else "")
    return {"messages": [
        {"role": "user", "content": user},
        {"role": "assistant", "content": row["output"]},
    ]}

alpaca_msgs = alpaca.map(alpaca_to_messages, remove_columns=alpaca.column_names)
train_ds = concatenate_datasets([train_ds, alpaca_msgs]).shuffle(seed=42)

print(f"train: {len(train_ds)} ({n_general} general + rest arcemx), eval: {len(eval_ds)}")

def format_chat(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = train_ds.map(format_chat)
eval_ds = eval_ds.map(format_chat)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=0.05,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=10,
        report_to="none",
        seed=42,
    ),
)

# eval-loss early stop: load_best_model_at_end above already reverts to
# the best eval-loss checkpoint post-training, which is the simplest
# correct form of early stopping at this dataset size/epoch count.
trainer_stats = trainer.train()
print(trainer_stats)

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
print(f"LoRA adapter saved to {OUTPUT_DIR}/lora_adapter")

In [ ]:
# Merge LoRA into the base weights and export a quantized GGUF for
# llama.cpp CPU inference (the serving path - see specialist_eval.py).
model.save_pretrained_gguf(
    f"{OUTPUT_DIR}/gguf",
    tokenizer,
    quantization_method=GGUF_QUANT,
)
print(f"GGUF exported under {OUTPUT_DIR}/gguf - find the .gguf file in this notebook's Output tab.")

## Next steps (manual, outside this notebook)

1. Download the `.gguf` file from the **Output** tab on the right.
2. Rename it `arcemx_specialist_v{N}.gguf` (increment N each training run).
3. Go to the arcemx GitHub repo -> Releases -> **Draft a new release**, tag it (e.g. `specialist-v1`), and attach the GGUF as a release asset (up to 2GB free).
4. Dispatch `.github/workflows/specialist_eval.yml` with that release tag as input.
5. Check the `/rankings` or accuracy dashboard after ~a week to see `specialist-v{N}` vs the live chain on each dimension.
6. The specialist stays advisory - nothing reads its output except grading - until its 30-day accuracy beats the live chain on >= 2 dimensions. **That promotion decision is yours, documented in ROADMAP.md, never automated.**